# 08 -- Alignment-Aware Positioning Analysis (Contact Luck v0.6)

Estimates whether the defense's STARTING alignment (not execution) changed the
expected outcome of a batted ball. Scoped as **alignment-aware positioning**, not
exact defender positioning -- the only public signal available is Statcast's coarse,
pre-pitch `if_fielding_alignment` / `of_fielding_alignment` labels, never exact
coordinates, pre-contact movement, reaction time, or route efficiency. See
`mlb_luck_score.models.compare_alignment_aware` module docstring and README.md
"Alignment-aware positioning (Version 0.6)" for the full scope and adoption rule.

Three controlled variants, IDENTICAL 2021-2023 training rows / 2024 validation rows,
`class_weight=None` throughout:

- `baseline_v02` -- current production model, unchanged.
- `alignment_labels_v06` -- baseline + raw `if_fielding_alignment`/`of_fielding_alignment`.
- `alignment_interactions_v06` -- labels + physically-motivated interaction terms.

`position_depth_v06` was investigated and is NOT implemented -- no reliable public
per-play/per-fielder positioning coordinate dataset was found (see module docstring).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/processed/cleaned_development_data_with_venue.parquet")
df = pd.read_parquet(DATA_PATH) if DATA_PATH.exists() else None
print(f"Loaded {len(df):,} rows" if df is not None else "Data not found -- run make join-venue-metadata first")

Loaded 494,173 rows


## 1. Alignment-label coverage

In [2]:
if df is not None:
    for col in ("if_fielding_alignment", "of_fielding_alignment"):
        print(f"--- {col} ---")
        print(f"non-null fraction: {df[col].notna().mean():.4f}")
        print(df[col].value_counts(dropna=False))
        print()
    print("Coverage by season:")
    print(df.groupby("season")["if_fielding_alignment"].apply(lambda s: s.notna().mean()))

--- if_fielding_alignment ---
non-null fraction: 0.9962
if_fielding_alignment
Standard         314587
Infield shift     76439
Infield shade     65179
Strategic         36108
NaN                1860
Name: count, dtype: int64

--- of_fielding_alignment ---
non-null fraction: 0.9962
of_fielding_alignment
Standard          467335
Strategic          24498
NaN                 1860
4th outfielder       480
Name: count, dtype: int64

Coverage by season:
season
2021    0.994371
2022    0.997232
2023    0.996820
2024    0.996483
Name: if_fielding_alignment, dtype: float64


## 2. Feature engineering: interaction terms

`add_alignment_interaction_features` computes, from the raw labels:

- `if_alignment_shift_indicator` / `of_alignment_shift_indicator`: 0.0 = `"Standard"`, 1.0 = any other alignment.
- `of_shift_x_launch_angle`, `of_shift_x_hit_distance`: outfield shift indicator x launch angle / projected distance.
- `if_shift_x_pull_groundball`: infield shift indicator AND pull-side AND ground ball.
- `if_alignment_x_stand`, `if_alignment_x_spray_sector`: joint categories (infield alignment x handedness / spray direction).

In [3]:
from mlb_luck_score.features.build_contact_features import add_alignment_interaction_features

if df is not None:
    sample = add_alignment_interaction_features(df.head(5))
    display_cols = [
        "if_fielding_alignment", "of_fielding_alignment", "if_alignment_shift_indicator",
        "of_alignment_shift_indicator", "of_shift_x_launch_angle", "if_shift_x_pull_groundball",
        "if_alignment_x_stand", "if_alignment_x_spray_sector",
    ]
    print(sample[display_cols])

  if_fielding_alignment of_fielding_alignment  if_alignment_shift_indicator  of_alignment_shift_indicator  of_shift_x_launch_angle  \
0         Infield shift              Standard                           1.0                           0.0                     -0.0   
1              Standard              Standard                           0.0                           0.0                      0.0   
2              Standard              Standard                           0.0                           0.0                     -0.0   
3              Standard              Standard                           0.0                           0.0                      0.0   
4              Standard              Standard                           0.0                           0.0                      0.0   

   if_shift_x_pull_groundball if_alignment_x_stand if_alignment_x_spray_sector  
0                         0.0      Infield shift_L   Infield shift_left_center  
1                         0.0    

## 3. Three-way model comparison

In [4]:
from mlb_luck_score.models.compare_alignment_aware import (
    ALIGNMENT_CANDIDATE_VARIANTS,
    run_alignment_aware_comparison,
)
from mlb_luck_score.models.compare_park_aware import VARIANT_BASELINE_V02

comparison = None
trained_models = None
proba_by_variant = None
if df is not None:
    comparison, trained_models, proba_by_variant = run_alignment_aware_comparison(df)
    rows = []
    for variant, summary in comparison.items():
        rows.append({
            "variant": variant,
            "log_loss": summary["multiclass_log_loss"],
            "ece": summary["expected_calibration_error"],
            "home_run_ece": summary["home_run_ece"],
            "accuracy(secondary)": summary["argmax_accuracy_secondary"],
            "n": summary["sample_count"],
        })
    print(pd.DataFrame(rows).set_index("variant"))

6 of 41 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 36 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 36 calibration bin(s) have fewer than 20 samples and are marked unreliable.


2 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


...

6 of 43 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 40 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 41 calibration bin(s) have fewer than 20 samples and are marked unreliable.


5 of 43 calibration bin(s) have fewer than 20 samples and are marked unreliable.


                            log_loss       ece  home_run_ece  accuracy(secondary)       n
variant                                                                                  
baseline_v02                0.670321  0.014413      0.002706             0.751457  122132
alignment_labels_v06        0.670287  0.014974      0.002918             0.751474  122132
alignment_interactions_v06  0.629065  0.013314      0.000912             0.738242  122132


## 4. Required evaluation-group calibration (batted-ball type, pull/oppo, shift, handedness, base state)

In [5]:
if comparison is not None:
    rows = []
    for variant in (VARIANT_BASELINE_V02, *ALIGNMENT_CANDIDATE_VARIANTS):
        for label, sub in comparison[variant]["alignment_subgroups"].items():
            rows.append({"variant": variant, "subgroup": label, "ece": sub["ece"], "n": sub["sample_count"]})
    subgroup_df = pd.DataFrame(rows).pivot(index="subgroup", columns="variant", values="ece")
    print(subgroup_df)

variant              alignment_interactions_v06  alignment_labels_v06  baseline_v02
subgroup                                                                           
bases_empty                            0.013496              0.015958      0.015503
bb_type_fly_ball                       0.014600              0.019394      0.018661
bb_type_ground_ball                    0.013671              0.008690      0.007763
bb_type_line_drive                     0.022313              0.035110      0.035155
bb_type_popup                          0.000780              0.001221      0.000927
lhb                                    0.010755              0.016013      0.015021
opposite_field                         0.018424              0.019700      0.019771
pull_side                              0.018813              0.028134      0.028340
rhb                                    0.016929              0.015806      0.015506
runners_on                             0.013985              0.014165      0

## 5. Per-venue calibration

Reused from `mlb_luck_score.models.compare_park_aware.compute_calibration_by_venue` --
only venues with at least the minimum reliable sample count are eligible to trigger the
material-regression rule.

In [6]:
if comparison is not None:
    for variant in ALIGNMENT_CANDIDATE_VARIANTS:
        venue_df = pd.DataFrame(comparison[variant]["calibration_by_venue"])
        reliable = venue_df[venue_df["reliable"]].sort_values("sample_count", ascending=False)
        print(f"--- {variant}: {len(reliable)} reliably-sampled venues ---")
        print(reliable.head(10))
        print()

--- alignment_labels_v06: 32 reliably-sampled venues ---
  venue_id  sample_count  ece_overall  ece_home_run  reliable
0       15          4359     0.018635      0.011727      True
1     4169          4304     0.015260      0.007898      True
2        7          4232     0.021058      0.014541      True
3     3309          4217     0.018927      0.010509      True
4       19          4216     0.022192      0.014157      True
5        2          4185     0.014959      0.008107      True
6       14          4164     0.019470      0.007384      True
7     2889          4152     0.020663      0.007473      True
8       31          4125     0.018324      0.009928      True
9        4          4122     0.016830      0.007243      True

--- alignment_interactions_v06: 32 reliably-sampled venues ---
  venue_id  sample_count  ece_overall  ece_home_run  reliable
0       15          4359     0.016431      0.010713      True
1     4169          4304     0.018598      0.004459      True
2        7 

## 6. Plays whose probabilities changed the most

Total-variation distance between `baseline_v02` and each candidate's predicted
probability vector, for qualitative review -- NOT itself an automated adoption
criterion (see module docstring).

In [7]:
if comparison is not None:
    for variant in ALIGNMENT_CANDIDATE_VARIANTS:
        print(f"--- {variant}: top 10 most-changed plays ---")
        most_changed = pd.DataFrame(comparison[variant]["most_changed_plays"][:10])
        cols = [c for c in most_changed.columns if not c.startswith(("baseline_p_", "candidate_p_"))]
        print(most_changed[cols])
        print()

--- alignment_labels_v06: top 10 most-changed plays ---
   probability_shift_tvd  game_pk  if_fielding_alignment  of_fielding_alignment     bb_type stand  spray_sector  is_pull  launch_angle  hit_distance_sc  \
0               0.128750   746520                    NaN                    NaN    fly_ball     R         right    False            30              380   
1               0.126628   745502                    NaN                    NaN    fly_ball     L         right     True            30              386   
2               0.124055   746561                    NaN                    NaN    fly_ball     L        center    False            38              371   
3               0.121013   746561                    NaN                    NaN    fly_ball     R        center    False            33              379   
4               0.118423   745164                    NaN                    NaN    fly_ball     L   left_center    False            37              366   
5             

## 7. Controlled-perturbation directional checks

1. Among real pull-side ground balls: does an "Infield shift" infield alignment predict
   FEWER singles (mean probability) than "Standard"? (Shifting exists specifically to
   convert pulled ground balls that would otherwise be singles.)
2. Among real pull-side fly balls: does a "Strategic" outfield alignment predict FEWER
   doubles than "Standard"?

In [8]:
from mlb_luck_score.models.compare_alignment_aware import (
    _prepare_alignment_columns,
    run_alignment_perturbation_checks,
)
from mlb_luck_score.models.compare_park_aware import _prepare_venue_column
from mlb_luck_score.config import VALIDATION_SEASONS

perturbation_by_candidate = {}
if comparison is not None:
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
    val_df = _prepare_venue_column(_prepare_alignment_columns(val_df))

    for variant in ALIGNMENT_CANDIDATE_VARIANTS:
        results = run_alignment_perturbation_checks(trained_models[variant], val_df)
        perturbation_by_candidate[variant] = results
        print(f"--- {variant} ---")
        for name, r in results.items():
            print(
                f"  {name}: {r.low_label}={r.mean_prob_low:.4f} {r.high_label}={r.mean_prob_high:.4f} "
                f"delta={r.delta:+.4f} expect_high_greater={r.expect_high_greater} passed={r.passed} n={r.sample_size}"
            )
        print()

--- alignment_labels_v06 ---
  infield_shift_reduces_pull_groundball_singles: Standard=0.2291 Infield shift=0.2150 delta=-0.0141 expect_high_greater=False passed=True n=34077
  outfield_strategic_reduces_pull_flyball_doubles: Standard=0.0566 Strategic=0.0606 delta=+0.0040 expect_high_greater=False passed=False n=11160



--- alignment_interactions_v06 ---
  infield_shift_reduces_pull_groundball_singles: Standard=0.2306 Infield shift=0.1509 delta=-0.0797 expect_high_greater=False passed=True n=34077
  outfield_strategic_reduces_pull_flyball_doubles: Standard=0.0596 Strategic=0.0697 delta=+0.0101 expect_high_greater=False passed=False n=11160



### Interpretation

The infield-shift check passes for both candidates with a real, sensible-sized effect:
shifted alignment materially lowers the model's predicted single probability for pulled
ground balls. The outfield-strategic check FAILS for both candidates -- "Strategic"
outfield alignment predicts MORE doubles, not fewer.

This is very plausibly **confounding by indication**, not a code bug: alignment is not
randomly assigned. Outfielders (and infielders) are more likely to shade/shift against
batters who are known extra-base threats in the first place, so a model trained on
observational data can learn "this alignment co-occurs with more doubles" (because of
who gets shifted against) rather than "this alignment causes more doubles." The
`bb_type_ground_ball` subgroup regression seen in Section 4 for
`alignment_interactions_v06` -- WORSE calibration on the very subgroup the shift
interaction targeted, even as aggregate log loss improved -- points at the same root
cause: the aggregate log-loss gain is plausibly driven by the outfield-distance
interaction terms picking up batter-power confounding on fly balls/line drives, not a
clean causal signal from infield-shift-vs-ground-balls. This is exactly the class of
result the task's adoption rule exists to catch.

## 8. Positioning counterfactual stability

In [9]:
from mlb_luck_score.models.compare_alignment_aware import check_positioning_counterfactual_stable

stability_by_candidate = {}
if comparison is not None:
    for variant in ALIGNMENT_CANDIDATE_VARIANTS:
        stability = check_positioning_counterfactual_stable(trained_models[variant], val_df)
        stability_by_candidate[variant] = stability
        print(variant, stability)

alignment_labels_v06 {'typical_predictions_well_formed': True, 'positioning_effect_zero_at_standard_alignment': True, 'max_abs_positioning_effect_at_standard_alignment': 0.0, 'stable': True}


alignment_interactions_v06 {'typical_predictions_well_formed': True, 'positioning_effect_zero_at_standard_alignment': True, 'max_abs_positioning_effect_at_standard_alignment': 0.0, 'stable': True}


## 9. Positioning-effect distribution (`alignment_interactions_v06`)

`positioning_effect = EV(actual alignment) - EV(typical alignment)`, holding every other
feature fixed. Positive = actual alignment made contact MORE favorable to the batter than
a typical ("Standard"/"Standard") alignment would have; negative = LESS favorable (the
shift/strategic alignment worked, from the batter's perspective).

**Not computed as an adopted scoring component here** -- neither v0.6 candidate passed
the adoption rule (see Section 10), so this is exploratory only, per the task's explicit
instruction to keep the positioning counterfactual separate from an actual adoption
decision.

In [10]:
from mlb_luck_score.features.build_contact_features import generate_typical_alignment_rows
from mlb_luck_score.models.train_contact_model import predict_proba_ordered
from mlb_luck_score.scoring.positioning_attribution import compute_positioning_attribution

if comparison is not None:
    trained = trained_models["alignment_interactions_v06"]
    feature_cols = trained.numeric_features + trained.categorical_features
    typical_val_df = generate_typical_alignment_rows(val_df)
    actual_proba = predict_proba_ordered(trained, val_df[feature_cols])
    typical_proba = predict_proba_ordered(trained, typical_val_df[feature_cols])
    attribution = compute_positioning_attribution(actual_proba, typical_proba)
    print(attribution["positioning_effect"].describe())
    print()
    print("Largest positive (actual alignment favored the batter most):")
    print(attribution.nlargest(5, "positioning_effect"))
    print()
    print("Largest negative (actual alignment hurt the batter most):")
    print(attribution.nsmallest(5, "positioning_effect"))

count    122132.000000
mean          0.000177
std           0.032323
min          -0.249145
25%           0.000000
50%           0.000000
75%           0.000000
max           0.198858
Name: positioning_effect, dtype: float64

Largest positive (actual alignment favored the batter most):
        expected_run_value_actual_alignment  expected_run_value_typical_alignment  positioning_effect
394987                             0.797981                              0.599123            0.198858
488201                             0.637120                              0.438678            0.198442
383379                             0.714841                              0.518202            0.196640
471786                             0.889610                              0.694135            0.195475
405266                             0.821757                              0.628217            0.193540

Largest negative (actual alignment hurt the batter most):
        expected_run_value_actual_alignmen

## 10. Adoption recommendation

In [11]:
from mlb_luck_score.models.compare_alignment_aware import (
    BOOTSTRAP_METRICS,
    compute_paired_bootstrap,
    recommend_alignment_adoption,
)

recommendation = None
if comparison is not None:
    y_true = val_df["outcome_class"].astype(str)
    bootstrap_by_candidate = {}
    for variant in ALIGNMENT_CANDIDATE_VARIANTS:
        bootstrap_by_candidate[variant] = compute_paired_bootstrap(
            y_true,
            proba_by_variant[VARIANT_BASELINE_V02],
            proba_by_variant[variant],
            val_df["game_pk"],
            metrics=BOOTSTRAP_METRICS,
        )
    recommendation = recommend_alignment_adoption(
        comparison, bootstrap_by_candidate, perturbation_by_candidate, stability_by_candidate
    )
    print(json.dumps(recommendation["per_candidate"], indent=2, default=str))
    print()
    print("recommend_adopt_any_v06_candidate:", recommendation["recommend_adopt_any_v06_candidate"])
    print("best_candidate:", recommendation["best_candidate"])

{
  "alignment_labels_v06": {
    "improves_log_loss": true,
    "log_loss_delta": -3.441348001009814e-05,
    "bootstrap_supports_improvement": false,
    "log_loss_bootstrap": {
      "metric": "log_loss",
      "point_estimate": -3.441348001009814e-05,
      "ci_low": -0.00012947357924969206,
      "ci_high": 7.408612041504551e-05,
      "confidence_level": 0.95,
      "n_reps": 500,
      "seed": 42,
      "resampling_unit": "game_pk"
    },
    "ece_not_materially_worse": true,
    "ece_delta": 0.0005614190289183127,
    "home_run_ece_not_materially_worse": true,
    "home_run_ece_delta": 0.00021131990501551496,
    "no_material_venue_regressions": true,
    "material_venue_regressions": [],
    "no_material_subgroup_regressions": true,
    "material_subgroup_regressions": [],
    "sufficient_alignment_coverage": true,
    "alignment_coverage_rate": 0.9968804244587823,
    "perturbation_checks_passed": false,
    "perturbation_failures": [
      "outfield_strategic_reduces_pull_fl

## 11. Explicit limitations

- **Alignment-aware POSITIONING, not exact defender positioning.** Only coarse, pre-pitch
  category labels are available publicly -- never exact coordinates, pre-contact movement,
  reaction time, route efficiency, or a judgment of whether the chosen alignment was
  strategically appropriate.
- **Alignment is not randomly assigned.** Defenses choose alignment based on batter
  tendencies (including power/pull profile), which means any model trained on
  observational alignment labels risks learning "who gets shifted against" rather than
  "what shifting causes" -- Section 7's outfield-strategic result is a concrete example.
- **`position_depth_v06` was not implemented.** No reliable public per-play/per-fielder
  coordinate or average-depth dataset was found to exist for joining -- this is a
  documented scope gap, not a decision to exclude numeric depth data on principle.
- **Neither candidate passed the Version 0.6 adoption rule** on real 2021-2024 data (see
  Section 10) -- `baseline_v02` remains the selected production model. The positioning
  counterfactual (Section 9) is exploratory infrastructure only, not an adopted Contact
  Luck component.
- This notebook uses only 2021-2024 data. 2025 remains untouched (see
  `mlb_luck_score.config.assert_seasons_allowed`).